In [4]:
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

# === 参数设置 ===
query = "dihydroxy acid dehydratase"
save_dir = r"C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase"
os.makedirs(save_dir, exist_ok=True)

# 可切换数据库
databases = ["UniProtKB", "EPO", "JPO", "USPTO"]

# === 获取 UniRef ID 列表（增加缓存机制） ===
def get_uniref_ids(db: str, query: str, page_size: int = 5000):
    cache_path = os.path.join(save_dir, f"{db}_ids.txt")

    # 若缓存存在，直接读取并返回
    if os.path.exists(cache_path):
        print(f"📁 发现缓存：{cache_path}，跳过 ID 重新下载")
        with open(cache_path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    print(f"🔍 获取 {db} ID 列表中 ...")
    ids = []
    start = 0

    while True:
        params = {"query": query, "format": "idlist", "size": page_size, "start": start}
        url = f"https://www.ebi.ac.uk/ebisearch/ws/rest/{db}"
        r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200:
            print(f"❌ 获取 {db} ID 失败：HTTP {r.status_code}")
            break

        new_ids = [line.strip() for line in r.text.splitlines() if line.strip()]
        if not new_ids:
            break

        ids.extend(new_ids)
        print(f"📥 已获取 {len(new_ids)} 条 (总计 {len(ids)})")

        if len(new_ids) < page_size:
            break

        start += page_size
        time.sleep(0.5)

    # 写入缓存
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write("\n".join(ids))

    return ids


# === 下载单条 FASTA ===
def fetch_fasta(uid, db_type):
    if db_type.lower() == "uniprotkb":
        url = f"https://rest.uniprot.org/uniprotkb/{uid}.fasta"
    else:
        url = f"https://rest.uniprot.org/uniref/{uid}.fasta"
    
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if r.status_code == 200 and r.text.startswith(">"):
            return r.text
    except:
        pass

    return None


# === 并行下载（加强续跑） ===
def download_uniref_parallel(db, ids, max_workers=10, progress_interval=50):
    db_dir = os.path.join(save_dir, db)
    os.makedirs(db_dir, exist_ok=True)

    # 检查已存在的 fasta
    existing = set(f[:-6] for f in os.listdir(db_dir) if f.endswith(".fasta"))

    # 检查是否存在失败记录
    failed_path = os.path.join(save_dir, f"{db}_failed_ids.txt")
    if os.path.exists(failed_path):
        with open(failed_path, "r", encoding="utf-8") as f:
            previous_failed = [line.strip() for line in f if line.strip()]
        print(f"🔁 发现之前下载失败的 {len(previous_failed)} 条，将优先重试")
        ids = previous_failed + [uid for uid in ids if uid not in existing and uid not in previous_failed]
    else:
        ids = [uid for uid in ids if uid not in existing]

    print(f"🔁 续跑模式：剩余 {len(ids)} 条未下载。")

    fail_ids = []
    success = 0
    total = len(ids)

    print(f"\n⬇️ 开始下载 {db.upper()} 的 FASTA 序列 ({total} 条 ID) ...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_fasta, uid, db): uid for uid in ids}
        
        for i, future in enumerate(as_completed(futures), 1):
            uid = futures[future]
            fasta = future.result()

            if fasta:
                with open(os.path.join(db_dir, f"{uid}.fasta"), "w", encoding="utf-8") as f:
                    f.write(fasta)
                success += 1
            else:
                fail_ids.append(uid)

            if i % progress_interval == 0 or i == total:
                print(f"📦 进度 {i}/{total} | 成功 {success} | 失败 {len(fail_ids)}")

    # 更新失败记录
    with open(failed_path, "w", encoding="utf-8") as f:
        f.write("\n".join(fail_ids))

    # 合并所有 fasta，避免重复
    merged_path = os.path.join(save_dir, f"{db.upper()}_merged.fasta")
    seq_set = set()
    with open(merged_path, "w", encoding="utf-8") as merged:
        for fname in sorted(os.listdir(db_dir)):
            if fname.endswith(".fasta"):
                with open(os.path.join(db_dir, fname), "r", encoding="utf-8") as f:
                    content = f.read().strip()
                    if content not in seq_set:
                        seq_set.add(content)
                        merged.write(content + "\n\n")

    print(f"✅ {db.upper()} 下载完成: 成功 {success} / 失败 {len(fail_ids)}")
    print(f"💾 合并文件保存于：{merged_path}\n")


# === 主程序 ===
for db in databases:
    ids = get_uniref_ids(db, query)
    if not ids:
        print(f"⚠️ {db} 无结果。")
        continue
    download_uniref_parallel(db, ids)

print("\n🎉 所有数据库下载完成！")


📁 发现缓存：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase\UniProtKB_ids.txt，跳过 ID 重新下载
🔁 发现之前下载失败的 15 条，将优先重试
🔁 续跑模式：剩余 15 条未下载。

⬇️ 开始下载 UNIPROTKB 的 FASTA 序列 (15 条 ID) ...
📦 进度 15/15 | 成功 15 | 失败 0
✅ UNIPROTKB 下载完成: 成功 15 / 失败 0
💾 合并文件保存于：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase\UNIPROTKB_merged.fasta

📁 发现缓存：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase\EPO_ids.txt，跳过 ID 重新下载
🔁 发现之前下载失败的 93 条，将优先重试
🔁 续跑模式：剩余 93 条未下载。

⬇️ 开始下载 EPO 的 FASTA 序列 (93 条 ID) ...
📦 进度 50/93 | 成功 0 | 失败 50
📦 进度 93/93 | 成功 0 | 失败 93
✅ EPO 下载完成: 成功 0 / 失败 93
💾 合并文件保存于：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase\EPO_merged.fasta

📁 发现缓存：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid dehydratase\JPO_ids.txt，跳过 ID 重新下载
🔁 发现之前下载失败的 2 条，将优先重试
🔁 续跑模式：剩余 2 条未下载。

⬇️ 开始下载 JPO 的 FASTA 序列 (2 条 ID) ...
📦 进度 2/2 | 成功 0 | 失败 2
✅ JPO 下载完成: 成功 0 / 失败 2
💾 合并文件保存于：C:\Users\localadmin\Desktop\其他重要\11.24\EBI\dihydroxy acid 